In [1]:
import numpy as np
import os

In [2]:
path = "C:/Users/hanle/Downloads/th14_mae_g_16_4/th14_mae_g_16_4"

In [3]:
for video in os.listdir(path):
    if video.endswith(".npy"):
        data = np.load(os.path.join(path, video))
        print(data.shape)
        # break

(250, 1408)
(499, 1408)
(3617, 1408)
(590, 1408)
(1555, 1408)
(1091, 1408)
(1701, 1408)
(1762, 1408)
(1331, 1408)
(867, 1408)
(853, 1408)
(109, 1408)
(738, 1408)
(1652, 1408)
(643, 1408)
(1436, 1408)
(976, 1408)
(1433, 1408)
(1261, 1408)
(4037, 1408)
(737, 1408)
(1841, 1408)
(766, 1408)
(349, 1408)
(1587, 1408)
(1570, 1408)
(1318, 1408)
(961, 1408)
(1610, 1408)
(1443, 1408)
(566, 1408)
(1748, 1408)
(1812, 1408)
(1038, 1408)
(1114, 1408)
(538, 1408)
(4841, 1408)
(948, 1408)
(1665, 1408)
(2201, 1408)
(910, 1408)
(6352, 1408)
(825, 1408)
(1375, 1408)
(1552, 1408)
(317, 1408)
(1566, 1408)
(1144, 1408)
(1034, 1408)
(1530, 1408)
(1159, 1408)
(2182, 1408)
(1174, 1408)
(750, 1408)
(210, 1408)
(927, 1408)
(5322, 1408)
(542, 1408)
(1783, 1408)
(1365, 1408)
(216, 1408)
(534, 1408)
(504, 1408)
(1504, 1408)
(372, 1408)
(657, 1408)
(1229, 1408)
(247, 1408)
(417, 1408)
(3236, 1408)
(1555, 1408)
(1336, 1408)
(1546, 1408)
(2448, 1408)
(1537, 1408)
(895, 1408)
(1911, 1408)
(233, 1408)
(641, 1408)
(2583,

In [4]:
from transformers import VideoMAEImageProcessor, AutoModel, AutoConfig
import numpy as np
import torch


config = AutoConfig.from_pretrained("OpenGVLab/VideoMAEv2-Large", trust_remote_code=True)
processor = VideoMAEImageProcessor.from_pretrained("OpenGVLab/VideoMAEv2-Large")
model = AutoModel.from_pretrained('OpenGVLab/VideoMAEv2-Large', config=config, trust_remote_code=True)


video = list(np.random.rand(16, 3, 224, 224))




# B, T, C, H, W -> B, C, T, H, W
inputs = processor(video, return_tensors="pt")
inputs['pixel_values'] = inputs['pixel_values'].permute(0, 2, 1, 3, 4)

with torch.no_grad():
  outputs = model(**inputs)

c:\phd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 316/316 [00:00<00:00, 11925.50it/s]


AttributeError: 'VideoMAEv2' object has no attribute 'all_tied_weights_keys'

In [5]:
import numpy as np
import torch
from transformers import VideoMAEImageProcessor, AutoModel, AutoConfig
import decord  # for fast video reading

config = AutoConfig.from_pretrained("OpenGVLab/VideoMAEv2-Large", trust_remote_code=True)
processor = VideoMAEImageProcessor.from_pretrained("OpenGVLab/VideoMAEv2-Large")
model = AutoModel.from_pretrained("OpenGVLab/VideoMAEv2-Large", config=config, trust_remote_code=True)
model.eval().cuda()

WINDOW = 16   # frames per window
STRIDE = 4    # frames between windows

def extract_features(video_path, target_fps=30):
    vr = decord.VideoReader(video_path)
    native_fps = vr.get_avg_fps()
    # resample frame indices to target_fps if needed
    frame_indices = np.arange(0, len(vr), native_fps / target_fps).astype(int)
    frames = vr.get_batch(frame_indices).asnumpy()  # [N, H, W, C]

    feats = []
    for start in range(0, len(frames) - WINDOW + 1, STRIDE):
        clip = frames[start:start + WINDOW]                 # [16, H, W, C]
        clip = list(clip)                                     # processor expects a list of frames
        inputs = processor(clip, return_tensors="pt")
        inputs["pixel_values"] = inputs["pixel_values"].permute(0, 2, 1, 3, 4).cuda()
        with torch.no_grad():
            out = model(**inputs)
        feats.append(out.last_hidden_state.mean(dim=1).cpu().numpy())  # pool to 1 vector

    return np.concatenate(feats, axis=0)  # [T, 1408]

Loading weights: 100%|██████████| 316/316 [00:00<00:00, 10229.77it/s]


AttributeError: 'VideoMAEv2' object has no attribute 'all_tied_weights_keys'